# ⏱️ Survival Analysis
**ISLP Chapter 11 · Pattern Recognition for the Rest of Us**

> Survival analysis answers questions like: how long until a patient relapses? When will a customer churn? It handles a unique challenge — censored observations: people who haven't experienced the event yet when the study ends.

### What you'll learn
- Censoring: why standard regression fails for time-to-event data
- Kaplan-Meier estimator: non-parametric survival curve
- Log-rank test: comparing survival between groups
- Cox proportional hazards model: regression for survival data
- Hazard ratio interpretation

### Dataset: ROSSI recidivism + simulated patient data
### Time: ~60 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
print('✓ Setup complete')
!pip install lifelines -q
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines.datasets import load_rossi
import warnings; warnings.filterwarnings('ignore')

# Rossi recidivism dataset: time until re-arrest after release from prison
rossi = load_rossi()
print(f'Rossi dataset: {rossi.shape}')
print(rossi.head())
print(f"\nEvent rate: {rossi['arrest'].mean():.1%} arrested")
print(f"Duration range: {rossi['week'].min()}–{rossi['week'].max()} weeks")

## 🎯 Part 1 — What is Censoring?

In time-to-event data:
- Some subjects experience the event (arrested, died, churned)
- Others are **censored** — we know they survived UNTIL a certain time, but not what happened after
  - Study ended before they experienced the event
  - Lost to follow-up
  - Withdrew from study

**Why standard regression fails:** A censored observation isn't a missing value — it contains real information (the person survived for at least X weeks). Ignoring it or treating it as missing biases estimates.

**Survival analysis** handles censoring properly.

In [ ]:
# Visualize censoring
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Show 20 individual timelines
np.random.seed(42)
sample_idx = np.random.choice(len(rossi), 25, replace=False)
for i, idx in enumerate(sample_idx):
    row = rossi.iloc[idx]
    color = '#e85d20' if row['arrest']==1 else '#1e5fa8'
    marker = 'x' if row['arrest']==1 else 'o'
    axes[0].plot([0, row['week']], [i, i], color=color, lw=1.5, alpha=0.8)
    axes[0].plot(row['week'], i, marker=marker, color=color, markersize=7)
axes[0].set_xlabel('Weeks'); axes[0].set_ylabel('Subject')
axes[0].set_title('Event timelines\n× = arrested (event), o = censored')
from matplotlib.lines import Line2D
legend = [Line2D([0],[0],color='#e85d20',marker='x',label='Arrested'),
          Line2D([0],[0],color='#1e5fa8',marker='o',label='Censored')]
axes[0].legend(handles=legend)

# Histogram of observation times by event status
for arrested, color, label in [(1,'#e85d20','Arrested'),(0,'#1e5fa8','Censored')]:
    subset = rossi[rossi['arrest']==arrested]['week']
    axes[1].hist(subset, bins=20, alpha=0.6, color=color, label=f'{label} (n={len(subset)})', edgecolor='white')
axes[1].set_xlabel('Weeks'); axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Follow-up Times')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Kaplan-Meier survival curve — overall
kmf = KaplanMeierFitter()
kmf.fit(rossi['week'], event_observed=rossi['arrest'], label='All subjects')

fig, ax = plt.subplots(figsize=(9, 5))
kmf.plot_survival_function(ax=ax, color='#1e5fa8', linewidth=2.5, ci_show=True, ci_alpha=0.15)
ax.set_xlabel('Weeks after release')
ax.set_ylabel('Probability of not being re-arrested')
ax.set_title('Kaplan-Meier Survival Curve — Recidivism Study')
ax.axhline(0.5, color='#888', lw=1, ls='--', label='Median survival')
median = kmf.median_survival_time_
if not np.isinf(median):
    ax.axvline(median, color='#888', lw=1, ls='--')
    ax.text(median+1, 0.52, f'Median = {median:.0f} weeks', fontsize=9)
ax.legend()
plt.tight_layout(); plt.show()
print(f'\nMedian survival time: {kmf.median_survival_time_:.1f} weeks')
print(f'52-week survival estimate: {kmf.survival_function_at_times([52]).values[0]:.3f}')

In [ ]:
# Compare groups: financial aid vs no financial aid
fig, ax = plt.subplots(figsize=(9, 5))
for group, color, label in [(1,'#1a7a45','Received financial aid'),(0,'#e85d20','No financial aid')]:
    kmf_g = KaplanMeierFitter()
    mask = rossi['fin']==group
    kmf_g.fit(rossi.loc[mask,'week'], event_observed=rossi.loc[mask,'arrest'], label=label)
    kmf_g.plot_survival_function(ax=ax, color=color, linewidth=2.5, ci_show=True, ci_alpha=0.15)
ax.set_xlabel('Weeks after release'); ax.set_ylabel('Probability of not being re-arrested')
ax.set_title('Kaplan-Meier by Financial Aid Status')
ax.legend()
plt.tight_layout(); plt.show()

# Log-rank test
results = logrank_test(
    rossi.loc[rossi['fin']==1,'week'], rossi.loc[rossi['fin']==0,'week'],
    event_observed_A=rossi.loc[rossi['fin']==1,'arrest'],
    event_observed_B=rossi.loc[rossi['fin']==0,'arrest']
)
print(f'Log-rank test: p-value = {results.p_value:.4f}')
print('Financial aid significantly reduces recidivism' if results.p_value < 0.05 else 'No significant difference')

In [ ]:
# Cox Proportional Hazards Model
cox_features = ['fin','age','race','wexp','mar','paro','prio']
cph = CoxPHFitter()
cph.fit(rossi[cox_features + ['week','arrest']], duration_col='week', event_col='arrest')
cph.print_summary()

# Plot hazard ratios
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.set_title('Cox Model: Hazard Ratios (HR > 1 = higher risk of arrest)')
ax.axvline(0, color='black', lw=0.8, ls='--')
plt.tight_layout(); plt.show()
print('\n📌 HR < 0 (log scale) = protective factor, HR > 0 = risk factor')
print(f"   'prio' (prior arrests): HR = {np.exp(cph.params_['prio']):.2f} — each prior arrest increases risk by {(np.exp(cph.params_['prio'])-1)*100:.0f}%")

In [ ]:
# Exercise workspace
# Task 1: Compare survival curves by age group (< 30 vs >= 30)
# YOUR CODE HERE

# Task 2: Build a Cox model using only the 3 most significant predictors
# Compare its concordance index to the full model
# YOUR CODE HERE

# Task 3: Predict individual survival curves for two specific profiles:
# Profile A: young (25), prior arrests=0, financial aid
# Profile B: older (40), prior arrests=3, no financial aid
profile_A = pd.DataFrame({'fin':1,'age':25,'race':1,'wexp':1,'mar':0,'paro':1,'prio':0}, index=[0])
profile_B = pd.DataFrame({'fin':0,'age':40,'race':1,'wexp':0,'mar':0,'paro':0,'prio':3}, index=[0])
# YOUR CODE HERE

In [ ]:
# @title 📝 Quiz — Survival Analysis
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What is censoring in survival analysis?
# @markdown **Q2:** What does the Kaplan-Meier curve show?
# @markdown **Q3:** What does the log-rank test compare?
# @markdown **Q4:** What does a hazard ratio of 2.0 mean in a Cox model?
# @markdown **Q5:** Why can't you use standard linear regression for time-to-event outcomes?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Survival Analysis"
# @title 🤖 AI Feedback — run this cell for instant grading
# @markdown **Enter your GitHub username below, then click ▶ Run**
# @markdown
# @markdown No API key needed — AI grading runs directly inside Colab for free.
# @markdown
# @markdown *(If AI grading is unavailable, you still get completion-based feedback)*

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

# ─────────────────────────────────────────────────────────────
import json, textwrap, re as _re

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _grade_with_gemini(answers_dict, nb_name):
    """Call Gemini inside Colab — no API key required."""
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    n_done   = len(answered)

    qa_lines = "\n".join(
        f"Q{k.replace('q','')}: {v}" for k, v in answered.items()
    )

    prompt = f"""You are a helpful TA grading quiz answers for the \"{nb_name}\" notebook
in a data science course called \"Data Pattern Recognition for the Rest of Us\",
based on ISLP (Introduction to Statistical Learning with Python).

Student answers ({n_done}/{n_total} answered):
{qa_lines if qa_lines else "(none provided)"}

Grade conceptual understanding, not exact wording. Accept reasonable paraphrases.
Be encouraging and specific. Reply ONLY in this JSON format, no markdown:
{{
  "quiz_score": <integer 0-{n_total}>,
  "grade": "<Excellent | Good | Needs Review | Incomplete>",
  "feedback": "<2-3 sentences: what they got right, any misconceptions>",
  "tip": "<one specific thing to remember from this technique>"
}}"""

    # ── Attempt 1: Colab-native Gemini (zero config) ──────────
    try:
        import google.generativeai as genai
        import google.auth
        # Use Colab's ambient credentials — no API key needed
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        genai.configure(credentials=creds)
        model  = genai.GenerativeModel("gemini-2.0-flash")
        result = model.generate_content(prompt)
        return result.text, "Gemini (Colab)"
    except Exception:
        pass

    # ── Attempt 2: Colab userdata key (if student added one) ──
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
        if api_key and len(api_key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=api_key)
            model  = genai.GenerativeModel("gemini-2.0-flash")
            result = model.generate_content(prompt)
            return result.text, "Gemini (API key)"
    except Exception:
        pass

    # ── Attempt 3: urllib fallback ─────────────────────────────
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY")
        if api_key and len(api_key) > 10:
            import urllib.request
            url = (f"https://generativelanguage.googleapis.com/v1beta/"
                   f"models/gemini-2.0-flash:generateContent?key={api_key}")
            payload = json.dumps({
                "contents": [{"parts": [{"text": prompt}]}],
                "generationConfig": {"maxOutputTokens": 1024, "temperature": 0.3}
            }).encode()
            req = urllib.request.Request(url, data=payload,
                                          headers={"Content-Type":"application/json"})
            with urllib.request.urlopen(req, timeout=30) as resp:
                data = json.loads(resp.read())
                return data["candidates"][0]["content"]["parts"][0]["text"], "Gemini (urllib)"
    except Exception:
        pass

    return None, None

def _rule_based_grade(answers_dict):
    n_total    = len(answers_dict)
    answered   = [v for v in answers_dict.values() if str(v).strip()]
    n_answered = len(answered)
    avg_len    = sum(len(str(v)) for v in answered) / max(n_answered, 1)
    score      = n_answered
    if n_answered == 0:
        return {"quiz_score":0,"grade":"Incomplete",
                "feedback":"No answers provided — fill in the quiz answers above and re-run.",
                "tip":"Each question only needs 1-2 sentences."}
    elif avg_len < 15:
        return {"quiz_score":score,"grade":"Needs Review",
                "feedback":f"You answered {n_answered}/{n_total} questions but responses are very brief. Try to explain your reasoning in 1-2 sentences.",
                "tip":"Aim for a sentence or two per answer — quality over brevity."}
    elif n_answered == n_total:
        return {"quiz_score":score,"grade":"Good",
                "feedback":f"All {n_total} questions answered with good detail.",
                "tip":"Review any concepts that felt unclear and check the Further Reading links."}
    else:
        return {"quiz_score":score,"grade":"Needs Review",
                "feedback":f"{n_answered} of {n_total} questions answered. Complete the remaining {n_total-n_answered} for full credit.",
                "tip":"Answer all questions before submitting."}

def _print_result(result, username, nb_name, source):
    colors = {"Excellent":"\033[92m","Good":"\033[94m",
              "Needs Review":"\033[93m","Incomplete":"\033[91m"}
    R     = "\033[0m"
    grade = result.get("grade","See feedback")
    C     = colors.get(grade,"\033[0m")
    n     = len(answers)
    qs    = result.get("quiz_score", 0)
    bar   = "\u2588"*qs + "\u2591"*(n-qs)

    print("\n" + "\u2550"*58)
    print(f"  \U0001f916  AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by  {source}")
    print("\u2550"*58)
    if username:
        print(f"  Student : @{username}")
    else:
        print(f"  Student : \u26a0\ufe0f  Enter your GitHub username in the box above")
    print(f"  Grade   : {C}{grade}{R}")
    print(f"  Score   : {qs}/{n}  [{bar}]")
    print()
    for line in textwrap.wrap(result.get("feedback",""), width=56):
        print(f"  {line}")
    tip = result.get("tip","")
    if tip:
        print()
        for line in textwrap.wrap(f"\U0001f4a1 {tip}", width=56):
            print(f"  {line}")
    print("\u2550"*58 + "\n")

# ── Main ──────────────────────────────────────────────────────
if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    n_answered = sum(1 for v in answers.values() if str(v).strip())
    username   = GITHUB_USERNAME.strip()

    print(f"  Notebook : {_NB_TITLE}")
    print(f"  Answers  : {n_answered}/{len(answers)} provided")
    if username: print(f"  Student  : @{username}")

    raw, source = _grade_with_gemini(answers, _NB_TITLE)

    if raw:
        try:
            clean  = _re.sub(r"```(?:json)?\s*|```","",raw).strip()
            result = json.loads(clean)
        except Exception:
            result = {"quiz_score":n_answered,
                      "grade":"Good" if n_answered==len(answers) else "Needs Review",
                      "feedback":raw[:400],"tip":""}
    else:
        print("  AI unavailable \u2014 using completion-based feedback\n")
        result = _rule_based_grade(answers)

    _print_result(result, username, _NB_TITLE, source)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


In [ ]:
_NB_NAME_ = "Survival Analysis"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username below so your instructor can track your submission, then click ▶ Run.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

import json, textwrap, re as _re
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _grade(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    qa       = "\n".join(f"Q{k.replace('q','')}: {v}" for k,v in answered.items())
    prompt   = (f"You are a TA grading quiz answers for \"{nb_name}\" in a data science "
                f"course (Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
                f"Student answers ({len(answered)}/{n_total}):\n{qa or '(none)'}\n\n"
                f"Grade conceptual understanding. Accept reasonable paraphrases. "
                f"Be encouraging and specific. Reply ONLY in this JSON (no markdown):\n"
                f"{{\"quiz_score\":<int 0-{n_total}>,"
                f"\"grade\":\"<Excellent|Good|Needs Review|Incomplete>\","
                f"\"feedback\":\"<2-3 sentences>\","
                f"\"tip\":\"<one takeaway>\"}}") 
    try:
        import google.generativeai as genai
        import google.auth, google.auth.transport.requests
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        genai.configure(credentials=creds)
        r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
        return r.text, "Gemini via Colab"
    except Exception:
        pass
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
            return r.text, "Gemini via key"
    except Exception:
        pass
    return None, None

def _fallback(answers_dict):
    n = len(answers_dict)
    done = [v for v in answers_dict.values() if str(v).strip()]
    nd, avg = len(done), sum(len(str(v)) for v in done)/max(len(done),1)
    if nd == 0: return {"quiz_score":0,"grade":"Incomplete","feedback":"No answers — fill in the quiz above.","tip":"Each question needs 1-2 sentences."}
    if avg < 15: return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered but very brief. Explain your reasoning.","tip":"Aim for 1-2 sentences per answer."}
    if nd == n:  return {"quiz_score":n,"grade":"Good","feedback":f"All {n} questions answered.","tip":"Review any concepts that felt unclear."}
    return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered. Complete the remaining {n-nd}.","tip":"Answer all questions for full credit."}

def _show(result, username, nb_name, source):
    C = {"Excellent":"\033[92m","Good":"\033[94m","Needs Review":"\033[93m","Incomplete":"\033[91m"}
    R = "\033[0m"; g = result.get("grade","?")
    n = len(answers); qs = result.get("quiz_score",0)
    print("\n"+"\u2550"*56)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*56)
    print(f"  Student : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade   : {C.get(g,'')} {g} {R}")
    print(f"  Score   : {qs}/{n}  [{'\u2588'*qs+'\u2591'*(n-qs)}]")
    print()
    [print(f"  {l}") for l in textwrap.wrap(result.get("feedback",""),54)]
    tip = result.get("tip","")
    if tip:
        print()
        [print(f"  \U0001f4a1 {l}") for l in textwrap.wrap(tip,52)]
    print("\u2550"*56+"\n")

if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd = sum(1 for v in answers.values() if str(v).strip())
    print(f"  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    raw, src = _grade(answers, _NB_TITLE)
    if raw:
        try:
            result = json.loads(_re.sub(r"```(?:json)?\s*|```","",raw).strip())
        except Exception:
            result = {"quiz_score":nd,"grade":"Good" if nd==len(answers) else "Needs Review","feedback":raw[:400],"tip":""}
    else:
        print("  (Gemini unavailable \u2014 using completion-based feedback)\n")
        result = _fallback(answers)
        src = None
    _show(result, GITHUB_USERNAME.strip(), _NB_TITLE, src)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")


## 📚 Further Reading
- [ISLP Ch 11](https://intro-stat-learning.github.io/ISLP/) — Survival analysis
- [lifelines docs](https://lifelines.readthedocs.io/)
- [Cox model interpretation guide](https://lifelines.readthedocs.io/en/latest/Survival%20Regression.html)

---
*Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*